# Chapter 12.4: Retrieval-Augmented Recommendation (RAG-Rec)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand how RAG extends recommendation with external knowledge
2. Build a retrieval-augmented recommender that integrates dynamic knowledge
3. Implement user review retrieval to enhance recommendation quality
4. Design product knowledge base augmentation for cold-start items
5. Compare RAG-based vs fine-tuning approaches for knowledge integration
6. Evaluate the impact of retrieval quality on recommendation performance
7. Handle real-time knowledge updates (news, trends, events)

## Prerequisites

- Understanding of embedding-based retrieval (Part 4)
- Familiarity with attention mechanisms and Transformers (Part 6)
- Knowledge of content-based filtering (Part 3)
- PyTorch proficiency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part12/chapter_12.4_rag_rec.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part12/chapter_12.4_rag_rec.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cpu')

print("All imports successful!")

## 1. Why RAG for Recommendation?

Traditional recommendation models rely solely on **learned embeddings** from interaction data.
RAG-Rec augments these with **retrieved external knowledge**:

$$\hat{y}_{u,i} = f(\mathbf{e}_u, \mathbf{e}_i, \text{Retrieve}(u, i, \mathcal{K}))$$

where $\mathcal{K}$ is an external knowledge base containing:
- User reviews and opinions
- Product descriptions and specifications
- News articles and trending topics
- Domain expertise and ontologies

> **💡 Concept:** RAG-Rec solves two key problems: (1) **cold-start** — new items have no interaction
> data but may have descriptions and reviews, and (2) **context awareness** — recommendations should
> adapt to current events (e.g., recommend umbrellas when rain is forecast).

In [ ]:
# Generate synthetic data with rich metadata
def generate_rag_data(n_users=300, n_items=200, n_reviews_per_item=5, knowledge_dim=32):
    """Generate users, items, reviews, and knowledge base."""
    # User embeddings
    user_embeddings = np.random.randn(n_users, knowledge_dim) * 0.3
    
    # Item embeddings and metadata
    item_embeddings = np.random.randn(n_items, knowledge_dim) * 0.3
    item_categories = np.random.randint(0, 10, n_items)
    
    # Simulate item descriptions as feature vectors
    item_descriptions = item_embeddings + np.random.randn(n_items, knowledge_dim) * 0.1
    
    # Simulate reviews as noisy item embeddings with sentiment
    reviews = []
    for item_id in range(n_items):
        for _ in range(n_reviews_per_item):
            reviewer_id = np.random.randint(0, n_users)
            # Review embedding = mix of user preference and item features
            review_emb = 0.5 * user_embeddings[reviewer_id] + 0.5 * item_embeddings[item_id]
            review_emb += np.random.randn(knowledge_dim) * 0.15
            sentiment = float(np.dot(user_embeddings[reviewer_id], item_embeddings[item_id]))
            sentiment = np.clip(sentiment + np.random.randn() * 0.3, -1, 1)
            reviews.append({
                'item_id': item_id,
                'reviewer_id': reviewer_id,
                'embedding': review_emb,
                'sentiment': sentiment
            })
    
    # Simulate trending topics (dynamic knowledge)
    trending_topics = np.random.randn(5, knowledge_dim) * 0.3  # 5 current trends
    
    # Generate interactions
    interactions = []
    for _ in range(5000):
        user_id = np.random.randint(0, n_users)
        item_id = np.random.randint(0, n_items)
        score = np.dot(user_embeddings[user_id], item_embeddings[item_id])
        # Add trend bonus
        trend_bonus = max(np.dot(item_embeddings[item_id], t) for t in trending_topics) * 0.2
        rating = np.clip(score + trend_bonus + np.random.randn() * 0.3 + 3.0, 1, 5)
        interactions.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': float(rating)
        })
    
    return {
        'user_embeddings': user_embeddings,
        'item_embeddings': item_embeddings,
        'item_descriptions': item_descriptions,
        'item_categories': item_categories,
        'reviews': reviews,
        'trending_topics': trending_topics,
        'interactions': interactions
    }

data = generate_rag_data()
print(f"Users: 300, Items: 200")
print(f"Reviews: {len(data['reviews'])}")
print(f"Interactions: {len(data['interactions'])}")
print(f"Trending topics: {data['trending_topics'].shape}")

## 2. The RAG-Rec Architecture

The RAG-Rec pipeline has three stages:

1. **Retrieve**: Find relevant knowledge for the (user, item) pair
2. **Augment**: Integrate retrieved knowledge into the representation
3. **Recommend**: Make the final prediction using augmented representation

$$\mathbf{k}_1, \ldots, \mathbf{k}_K = \text{Retrieve}(\mathbf{q}_{u,i}, \mathcal{K})$$

$$\mathbf{h}_{\text{aug}} = \text{CrossAttention}(\mathbf{q}_{u,i}, [\mathbf{k}_1, \ldots, \mathbf{k}_K])$$

$$\hat{y} = \text{MLP}([\mathbf{e}_u; \mathbf{e}_i; \mathbf{h}_{\text{aug}}])$$

In [ ]:
class KnowledgeRetriever:
    """Retrieve relevant knowledge for a (user, item) pair."""
    
    def __init__(self, reviews, item_descriptions, trending_topics, knowledge_dim=32):
        self.knowledge_dim = knowledge_dim
        
        # Build review index (item_id -> list of review embeddings)
        self.review_index = defaultdict(list)
        for review in reviews:
            self.review_index[review['item_id']].append({
                'embedding': review['embedding'],
                'sentiment': review['sentiment']
            })
        
        self.item_descriptions = item_descriptions  # (n_items, dim)
        self.trending_topics = trending_topics  # (n_topics, dim)
    
    def retrieve(self, user_embedding, item_id, top_k=3):
        """Retrieve top-K relevant knowledge pieces."""
        retrieved = []
        
        # 1. Item description (always included)
        retrieved.append(self.item_descriptions[item_id])
        
        # 2. Most relevant reviews for this item
        item_reviews = self.review_index.get(item_id, [])
        if item_reviews:
            # Score reviews by relevance to user
            scores = [np.dot(user_embedding, r['embedding']) for r in item_reviews]
            top_indices = np.argsort(scores)[-top_k:]
            for idx in top_indices:
                retrieved.append(item_reviews[idx]['embedding'])
        
        # 3. Most relevant trending topic
        topic_scores = self.trending_topics @ user_embedding
        best_topic = np.argmax(topic_scores)
        retrieved.append(self.trending_topics[best_topic])
        
        # Pad/truncate to fixed size
        while len(retrieved) < top_k + 2:
            retrieved.append(np.zeros(self.knowledge_dim))
        retrieved = retrieved[:top_k + 2]
        
        return np.array(retrieved)

retriever = KnowledgeRetriever(
    data['reviews'], data['item_descriptions'], data['trending_topics']
)

# Test retrieval
user_emb = data['user_embeddings'][0]
knowledge = retriever.retrieve(user_emb, item_id=5)
print(f"Retrieved knowledge shape: {knowledge.shape}")  # (5, 32)

In [ ]:
class RAGRecModel(nn.Module):
    """Retrieval-Augmented Recommendation Model."""
    
    def __init__(self, n_users=300, n_items=200, d_model=32, n_knowledge=5,
                 n_heads=4, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        
        # User and item embeddings
        self.user_embedding = nn.Embedding(n_users, d_model)
        self.item_embedding = nn.Embedding(n_items, d_model)
        
        # Knowledge encoder
        self.knowledge_proj = nn.Linear(d_model, d_model)
        
        # Cross-attention: query=(user,item), key/value=knowledge
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(d_model)
        
        # Prediction head
        self.predictor = nn.Sequential(
            nn.Linear(d_model * 3, hidden_dim),  # user + item + augmented
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, user_ids, item_ids, knowledge):
        """
        user_ids: (batch,)
        item_ids: (batch,)
        knowledge: (batch, n_knowledge, d_model)
        """
        user_emb = self.user_embedding(user_ids)  # (batch, d_model)
        item_emb = self.item_embedding(item_ids)  # (batch, d_model)
        
        # Project knowledge
        k_proj = self.knowledge_proj(knowledge)  # (batch, n_knowledge, d_model)
        
        # Query = user-item interaction representation
        query = (user_emb + item_emb).unsqueeze(1)  # (batch, 1, d_model)
        
        # Cross-attention over retrieved knowledge
        augmented, attention_weights = self.cross_attention(
            query, k_proj, k_proj
        )  # (batch, 1, d_model)
        augmented = self.layer_norm(augmented.squeeze(1) + user_emb + item_emb)
        
        # Concatenate and predict
        combined = torch.cat([user_emb, item_emb, augmented], dim=-1)
        rating_pred = self.predictor(combined).squeeze(-1)
        
        return rating_pred, attention_weights

rag_model = RAGRecModel()
print(f"RAG-Rec Model: {sum(p.numel() for p in rag_model.parameters()):,} parameters")

In [ ]:
# Prepare dataset with retrieved knowledge
class RAGRecDataset(Dataset):
    def __init__(self, interactions, retriever, user_embeddings):
        self.interactions = interactions
        self.retriever = retriever
        self.user_embeddings = user_embeddings
    
    def __len__(self):
        return len(self.interactions)
    
    def __getitem__(self, idx):
        inter = self.interactions[idx]
        user_emb = self.user_embeddings[inter['user_id']]
        knowledge = self.retriever.retrieve(user_emb, inter['item_id'])
        return {
            'user_id': torch.tensor(inter['user_id'], dtype=torch.long),
            'item_id': torch.tensor(inter['item_id'], dtype=torch.long),
            'rating': torch.tensor(inter['rating'], dtype=torch.float),
            'knowledge': torch.tensor(knowledge, dtype=torch.float)
        }

split = int(0.8 * len(data['interactions']))
train_inter = data['interactions'][:split]
val_inter = data['interactions'][split:]

train_ds = RAGRecDataset(train_inter, retriever, data['user_embeddings'])
val_ds = RAGRecDataset(val_inter, retriever, data['user_embeddings'])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# Train RAG-Rec model
def train_rag_model(model, train_loader, val_loader, n_epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            pred, _ = model(batch['user_id'], batch['item_id'], batch['knowledge'])
            loss = criterion(pred, batch['rating'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                pred, _ = model(batch['user_id'], batch['item_id'], batch['knowledge'])
                val_loss += criterion(pred, batch['rating']).item()
        
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} - Train: {total_loss/len(train_loader):.4f}, "
                  f"Val: {val_loss/len(val_loader):.4f}")
    return history

rag_model = RAGRecModel()
rag_history = train_rag_model(rag_model, train_loader, val_loader)

## 3. Baseline: Non-RAG Model

For comparison, we train a standard collaborative filtering model without knowledge retrieval.

> **🔑 Pro Tip:** Always compare RAG with a strong non-RAG baseline. The retrieval overhead
> is only justified if it provides measurable improvement.

In [ ]:
class BaseRecModel(nn.Module):
    """Standard collaborative filtering model (no RAG)."""
    def __init__(self, n_users=300, n_items=200, d_model=32, hidden_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, d_model)
        self.item_embedding = nn.Embedding(n_items, d_model)
        self.predictor = nn.Sequential(
            nn.Linear(d_model * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, user_ids, item_ids, knowledge=None):
        user_emb = self.user_embedding(user_ids)
        item_emb = self.item_embedding(item_ids)
        combined = torch.cat([user_emb, item_emb], dim=-1)
        return self.predictor(combined).squeeze(-1), None

base_model = BaseRecModel()
base_history = train_rag_model(base_model, train_loader, val_loader)

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rag_history['val_loss'], label='RAG-Rec', linewidth=2)
ax.plot(base_history['val_loss'], label='Base Model (no RAG)', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation MSE')
ax.set_title('RAG-Rec vs Standard Collaborative Filtering')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final Val Loss - RAG: {rag_history['val_loss'][-1]:.4f}, Base: {base_history['val_loss'][-1]:.4f}")

## 4. Attention Analysis: What Knowledge Matters?

One advantage of the cross-attention mechanism is interpretability. We can inspect the
attention weights to understand which knowledge sources are most important.

In [ ]:
# Analyze attention patterns
rag_model.eval()
knowledge_labels = ['Description', 'Review 1', 'Review 2', 'Review 3', 'Trend']

all_attention_weights = []
with torch.no_grad():
    for batch in val_loader:
        _, attn = rag_model(batch['user_id'], batch['item_id'], batch['knowledge'])
        all_attention_weights.append(attn.squeeze(1).numpy())

all_attn = np.concatenate(all_attention_weights, axis=0)  # (n_samples, n_knowledge)
mean_attn = all_attn.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average attention distribution
bars = axes[0].bar(knowledge_labels, mean_attn, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'],
                    edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Average Attention Weight')
axes[0].set_title('Knowledge Source Importance')
axes[0].grid(True, alpha=0.3, axis='y')

# Attention distribution heatmap (sample of users)
sample_attn = all_attn[:20]
im = axes[1].imshow(sample_attn, aspect='auto', cmap='YlOrRd')
axes[1].set_xticks(range(len(knowledge_labels)))
axes[1].set_xticklabels(knowledge_labels, rotation=45)
axes[1].set_ylabel('Sample')
axes[1].set_title('Attention Weights per Sample')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 5. RAG for Cold-Start Items

RAG particularly shines for **cold-start items** that have descriptions and reviews but no
interaction data. The model can leverage retrieved knowledge to make recommendations
without any collaborative filtering signal.

> **💡 Concept:** For cold-start items, RAG essentially converts the problem from collaborative
> filtering (where the item has no signal) to content-based filtering (using retrieved
> descriptions and similar items' reviews).

In [ ]:
def evaluate_cold_start(rag_model, base_model, data, retriever, n_cold_items=20):
    """Evaluate models on cold-start items."""
    # Items with very few interactions
    item_counts = defaultdict(int)
    for inter in data['interactions']:
        item_counts[inter['item_id']] += 1
    
    cold_items = sorted(item_counts.keys(), key=lambda x: item_counts[x])[:n_cold_items]
    warm_items = sorted(item_counts.keys(), key=lambda x: item_counts[x], reverse=True)[:n_cold_items]
    
    results = {'rag_cold': [], 'base_cold': [], 'rag_warm': [], 'base_warm': []}
    
    rag_model.eval()
    base_model.eval()
    
    with torch.no_grad():
        for inter in data['interactions']:
            uid = torch.tensor([inter['user_id']], dtype=torch.long)
            iid = torch.tensor([inter['item_id']], dtype=torch.long)
            user_emb = data['user_embeddings'][inter['user_id']]
            knowledge = torch.tensor(
                retriever.retrieve(user_emb, inter['item_id']), dtype=torch.float
            ).unsqueeze(0)
            
            rag_pred, _ = rag_model(uid, iid, knowledge)
            base_pred, _ = base_model(uid, iid, knowledge)
            error_rag = abs(rag_pred.item() - inter['rating'])
            error_base = abs(base_pred.item() - inter['rating'])
            
            if inter['item_id'] in cold_items:
                results['rag_cold'].append(error_rag)
                results['base_cold'].append(error_base)
            elif inter['item_id'] in warm_items:
                results['rag_warm'].append(error_rag)
                results['base_warm'].append(error_base)
    
    return {k: np.mean(v) if v else float('nan') for k, v in results.items()}

cs_results = evaluate_cold_start(rag_model, base_model, data, retriever)
print("Cold-Start Item Performance (MAE):")
print(f"  RAG-Rec: {cs_results['rag_cold']:.4f}")
print(f"  Base:    {cs_results['base_cold']:.4f}")
print("\nWarm Item Performance (MAE):")
print(f"  RAG-Rec: {cs_results['rag_warm']:.4f}")
print(f"  Base:    {cs_results['base_warm']:.4f}")

## 🏋️ Exercise 1: Implement Dynamic Knowledge Update

Simulate how recommendations change when the knowledge base is updated in real-time
(e.g., a new trending topic emerges).

In [ ]:
# TODO: Implement dynamic knowledge update
def simulate_knowledge_update(rag_model, retriever, user_id, item_id, user_embeddings):
    """Show how predictions change when knowledge is updated."""
    # TODO:
    # 1. Get prediction with current knowledge
    # 2. Update trending_topics in retriever
    # 3. Get prediction with updated knowledge
    # 4. Compare and visualize the difference
    pass

print("Exercise 1: Implement dynamic knowledge update")

## 🏋️ Exercise 2: Implement Retrieval Quality Ablation

Study how recommendation quality degrades as retrieval quality degrades.
Add increasing noise to the retrieved knowledge and measure the impact.

In [ ]:
# TODO: Retrieval quality ablation
def retrieval_quality_ablation(rag_model, val_loader, noise_levels):
    """Measure recommendation quality as retrieval quality degrades."""
    # TODO:
    # For each noise level:
    #   1. Add Gaussian noise to retrieved knowledge
    #   2. Evaluate model performance
    #   3. Record the degradation
    # Plot noise level vs MSE
    pass

print("Exercise 2: Implement retrieval quality ablation")

## 🏋️ Exercise 3: RAG vs Fine-Tuning Comparison

Compare RAG (retrieving knowledge at inference) vs fine-tuning (encoding knowledge into model weights)
for incorporating new item information.

In [ ]:
# TODO: Compare RAG vs fine-tuning
# 1. Train a base model on all interactions
# 2. When new items arrive:
#    a. RAG approach: retrieve descriptions/reviews, no retraining
#    b. Fine-tuning approach: retrain with new item data
# 3. Compare speed and accuracy on new items

print("Exercise 3: Compare RAG vs fine-tuning approaches")

## Summary

In this notebook, we explored Retrieval-Augmented Recommendation:

1. **RAG-Rec architecture**: Retrieve-Augment-Recommend pipeline with cross-attention
2. **Knowledge sources**: Reviews, descriptions, trending topics
3. **Cold-start advantage**: RAG helps when items have no interaction history
4. **Attention analysis**: Understanding which knowledge sources matter most
5. **Dynamic knowledge**: Updating recommendations as knowledge changes

### Key Takeaways

- RAG-Rec bridges collaborative filtering and content-based approaches
- Cross-attention provides interpretable knowledge integration
- Cold-start items benefit most from RAG augmentation
- Retrieval quality directly impacts recommendation quality
- RAG allows updating knowledge without retraining the model